In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
from pathlib import Path

PREPROCESSED_ROOT = Path(
    "/kaggle/input/datasets/mjayanth007/aptos-preprocessed/aptos_preprocessed"
)

TRAIN_DIR = PREPROCESSED_ROOT / "train"
TEST_DIR = PREPROCESSED_ROOT / "test"
LABELS_CSV = PREPROCESSED_ROOT / "train_labels.csv"

print("Train directory exists:", TRAIN_DIR.exists())
print("Test directory exists:", TEST_DIR.exists())
print("Labels CSV exists:", LABELS_CSV.exists())

Train directory exists: True
Test directory exists: True
Labels CSV exists: True


In [3]:
import pandas as pd
from pathlib import Path

labels_df = pd.read_csv(LABELS_CSV)

print("Label dataset shape:", labels_df.shape)
print("Columns:", labels_df.columns.tolist())
display(labels_df.head())

print("\nLabel distribution:")
print(labels_df.iloc[:, -1].value_counts().sort_index())

print("\nNumber of training files:", sum(1 for p in TRAIN_DIR.rglob("*") if p.is_file()))
print("Number of test files:", sum(1 for p in TEST_DIR.rglob("*") if p.is_file()))

Label dataset shape: (3662, 2)
Columns: ['id_code', 'diagnosis']


,id_code,diagnosis
0,000c1434d8d7,2
1,001639a390f0,4
2,0024cdab0c1e,1
3,002c21358ce6,0
4,005b95c28852,0



Label distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

Number of training files: 3662
Number of test files: 1928


In [4]:
from collections import Counter
from PIL import Image

image_paths = sorted(
    p for p in TRAIN_DIR.rglob("*")
    if p.is_file()
)

print("Image count:", len(image_paths))

print("\nImage extensions:")
print(Counter(p.suffix.lower() for p in image_paths))

print("\nSample image details:")
for path in image_paths[:5]:
    with Image.open(path) as img:
        print(
            path.name,
            "| size:", img.size,
            "| mode:", img.mode
        )

Image count: 3662

Image extensions:
Counter({'.jpg': 3662})

Sample image details:
000c1434d8d7.jpg | size: (512, 512) | mode: RGB
001639a390f0.jpg | size: (512, 512) | mode: RGB
0024cdab0c1e.jpg | size: (512, 512) | mode: RGB
002c21358ce6.jpg | size: (512, 512) | mode: RGB
005b95c28852.jpg | size: (512, 512) | mode: RGB


In [5]:
from sklearn.model_selection import train_test_split
from pathlib import Path

IMAGE_SIZE = 224
BATCH_SIZE = 32
RANDOM_SEED = 42
NUM_CLASSES = 5

labels_df["image_path"] = labels_df["id_code"].apply(
    lambda image_id: str(TRAIN_DIR / f"{image_id}.jpg")
)

# Verify that every label maps to an existing image
labels_df["file_exists"] = labels_df["image_path"].apply(
    lambda path: Path(path).is_file()
)

print("Missing images:", (~labels_df["file_exists"]).sum())

assert labels_df["file_exists"].all(), "Some labeled images are missing."
assert labels_df["diagnosis"].between(0, 4).all(), "Unexpected labels found."

train_df, val_df = train_test_split(
    labels_df.drop(columns="file_exists"),
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=labels_df["diagnosis"]
)

print("\nTraining images:", len(train_df))
print("Validation images:", len(val_df))

print("\nTraining grade distribution:")
print(train_df["diagnosis"].value_counts().sort_index())

print("\nValidation grade distribution:")
print(val_df["diagnosis"].value_counts().sort_index())

Missing images: 0

Training images: 2929
Validation images: 733

Training grade distribution:
diagnosis
0    1444
1     296
2     799
3     154
4     236
Name: count, dtype: int64

Validation grade distribution:
diagnosis
0    361
1     74
2    200
3     39
4     59
Name: count, dtype: int64


In [6]:
import tensorflow as tf
import numpy as np

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE

def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])

    # ResNet50 ImageNet preprocessing
    image = tf.keras.applications.resnet50.preprocess_input(image)

    label = tf.cast(label, tf.int32)
    return image, label

def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["diagnosis"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())

for images, labels in train_ds.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Labels in batch:", labels.numpy())

I0000 00:00:1791968238.378320      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791968238.386045      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Training batches: 92
Validation batches: 23
Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Image dtype: <dtype: 'float32'>
Labels in batch: [0 2 0 0 3 0 4 3 4 0 2 0 4 0 0 2 4 0 2 0 0 0 0 2 0 0 2 0 0 0 2 2]


In [7]:
import tensorflow as tf

NUM_CLASSES = 5

base_model = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

resnet_model = tf.keras.Model(inputs, outputs)

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

resnet_model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,597,957 (90.02 MB)

 Trainable params: 10,245 (40.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [8]:
from pathlib import Path
import tensorflow as tf

OUTPUT_DIR = Path("/kaggle/working/resnet_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "resnet50_best.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
 1/92 ━━━━━━━━━━━━━━━━━━━━ 20:16 13s/step - accuracy: 0.1562 - loss: 2.2682

I0000 00:00:1791968257.067096      68 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.5592 - loss: 1.2433
Epoch 1: val_loss improved from None to 0.74159, saving model to /kaggle/working/resnet_outputs/resnet50_best.keras

Epoch 1: finished saving model to /kaggle/working/resnet_outputs/resnet50_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 35s 239ms/step - accuracy: 0.6463 - loss: 0.9976 - val_accuracy: 0.7312 - val_loss: 0.7416 - learning_rate: 0.0010
Epoch 2/10
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.7180 - loss: 0.7570
Epoch 2: val_loss improved from 0.74159 to 0.70137, saving model to /kaggle/working/resnet_outputs/resnet50_best.keras

Epoch 2: finished saving model to /kaggle/working/resnet_outputs/resnet50_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 104ms/step - accuracy: 0.7204 - loss: 0.7453 - val_accuracy: 0.7435 - val_loss: 0.7014 - learning_rate: 0.0010
Epoch 3/10
91/92 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.7462 - loss: 0.6595
Epoch 3: val_loss did not improve from 0.70137
92/92 ━━━━━━━━━━━

In [9]:
# Freeze the entire backbone first
base_model.trainable = False

# Unfreeze only the final convolutional block
for layer in base_model.layers:
    if layer.name.startswith("conv5_"):
        layer.trainable = True

# Keep Batch Normalization frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Backbone configuration completed.")

print(
    "Trainable parameters:",
    sum(
        int(tf.keras.backend.count_params(w))
        for w in resnet_model.trainable_weights
    )
)

print("Fine-tuning setup ready.")

Backbone configuration completed.
Trainable parameters: 14963717
Fine-tuning setup ready.


In [10]:
resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model recompiled for fine-tuning.")

Model recompiled for fine-tuning.


In [11]:
FINE_TUNED_MODEL_PATH = (
    OUTPUT_DIR / "resnet50_finetuned_best.keras"
)

finetune_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(FINE_TUNED_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_finetune = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=finetune_callbacks
)

Epoch 1/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.8169 - loss: 0.4860
Epoch 1: val_loss improved from None to 0.59406, saving model to /kaggle/working/resnet_outputs/resnet50_finetuned_best.keras

Epoch 1: finished saving model to /kaggle/working/resnet_outputs/resnet50_finetuned_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 39s 274ms/step - accuracy: 0.8197 - loss: 0.4823 - val_accuracy: 0.7913 - val_loss: 0.5941 - learning_rate: 1.0000e-05
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.8779 - loss: 0.3469
Epoch 2: val_loss improved from 0.59406 to 0.58906, saving model to /kaggle/working/resnet_outputs/resnet50_finetuned_best.keras

Epoch 2: finished saving model to /kaggle/working/resnet_outputs/resnet50_finetuned_best.keras
92/92 ━━━━━━━━━━━━━━━━━━━━ 14s 154ms/step - accuracy: 0.8628 - loss: 0.3737 - val_accuracy: 0.7954 - val_loss: 0.5891 - learning_rate: 1.0000e-05
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.8978 - loss: 0.2992
Epo

In [12]:
import tensorflow as tf
from pathlib import Path

BEST_MODEL_PATH = Path(
    "/kaggle/working/resnet_outputs/resnet50_finetuned_best.keras"
)

best_resnet = tf.keras.models.load_model(BEST_MODEL_PATH)

print("Best model loaded successfully.")
print("Model output shape:", best_resnet.output_shape)
print("Checkpoint exists:", BEST_MODEL_PATH.exists())

Best model loaded successfully.
Model output shape: (None, 5)
Checkpoint exists: True


In [13]:
val_loss, val_accuracy = best_resnet.evaluate(
    val_ds,
    verbose=1
)

print(f"\nValidation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Validation accuracy (%): {val_accuracy * 100:.2f}%")

23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 169ms/step - accuracy: 0.7954 - loss: 0.5891

Validation loss: 0.5891
Validation accuracy: 0.7954
Validation accuracy (%): 79.54%


In [14]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

y_prob = best_resnet.predict(val_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("Classification report:\n")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=[
        "No DR (0)",
        "Mild (1)",
        "Moderate (2)",
        "Severe (3)",
        "Proliferative DR (4)"
    ],
    zero_division=0
))

print("Confusion matrix:\n")
print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4]))

23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 234ms/step
Classification report:

                      precision    recall  f1-score   support

           No DR (0)       0.94      0.97      0.95       361
            Mild (1)       0.52      0.35      0.42        74
        Moderate (2)       0.70      0.84      0.77       200
          Severe (3)       0.57      0.33      0.42        39
Proliferative DR (4)       0.54      0.42      0.48        59

            accuracy                           0.80       733
           macro avg       0.65      0.58      0.61       733
        weighted avg       0.78      0.80      0.78       733

Confusion matrix:

[[350  10   1   0   0]
 [ 13  26  33   0   2]
 [  8   8 169   7   8]
 [  1   1  13  13  11]
 [  1   5  25   3  25]]


In [15]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

OUTPUT_DIR = Path("/kaggle/working/resnet_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

with open(OUTPUT_DIR / "resnet_classification_report.json", "w") as f:
    json.dump(report, f, indent=2)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])

pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
).to_csv(OUTPUT_DIR / "resnet_confusion_matrix.csv")

print("Saved classification report.")
print("Saved confusion matrix.")
print("Best model:", OUTPUT_DIR / "resnet50_finetuned_best.keras")

Saved classification report.
Saved confusion matrix.
Best model: /kaggle/working/resnet_outputs/resnet50_finetuned_best.keras
